#  <span style="color:orange">Detección del Fraude. Nivel avanzado</span>

Fuente: https://pycaret.org/

En este tutorial usaremos el módulo `pycaret.classification` para aprender:

- **Normalización:** Cómo normalizar y escalar el conjunto de datos
- **Transformación:** Cómo aplicar transformaciones que hacen que los datos sean lineales y con distribucion normal
- **Ignorar la varianza baja:** Cómo eliminar características con variaciones estadísticamente insignificantes para hacer que el modelo sea más eficiente
- **Eliminar multicolinealidad:** Cómo eliminar multicolinealidad del conjunto de datos para mejorar el rendimiento de los algoritmos lineales
- **Funciones de grupo:** Cómo extraer información estadística de funciones relacionadas en el conjunto de datos
- **Agrupar variables numéricas:** Cómo agrupar variables numéricas y transformar características numéricas en categóricas'
- **Ensamblaje y stacking de modelos:** Cómo aumentar el rendimiento del modelo utilizando varias técnicas de ensamblaje, como Bagging, Boosting, Soft/hard Voting and Generalized Stacking
- **Calibración del modelo:** Cómo calibrar las probabilidades de un modelo de clasificación
- **Registro de experimentos:** Cómo registrar experimentos en PyCaret usando el backend MLFlow

## Explicación de las técnicas empleadas

A coninuación se describen el conjunto de tecnicas aplicadas al preprocesamiento de datos:

- **Normalización:** se usa para transformar o estandarizar los valores reales de las variables numéricas. Muchos algoritmos, como Regresión logística, Máquina de vectores de soporte, K Vecinos y Bayes, asumen que todas las entidades están centradas alrededor de cero y tienen variaciones que están en el mismo nivel de orden. Si una característica particular en un conjunto de datos tiene una varianza que es mayor en orden de magnitud que otras características, es posible que el modelo no comprenda todas las características correctamente y podría funcionar deficientemente. Por ejemplo, en el conjunto de datos que estamos usando para este ejemplo, la característica "AGE" varía entre 21 y 79, mientras que otras características numéricas varían entre 10,000 y 1,000,000.<br/>
<br/>
- **Transformación:** Si bien la normalización transforma el rango de datos para eliminar el impacto de la magnitud en la varianza, la transformación cambia la forma de la distribución para que los datos transformados puedan ser representados por una normal. En general, se debe transformar los datos si se usa algoritmos que asumen normalidad o una distribución gaussiana. Ejemplos de tales modelos son Regresión logística, Análisis discriminante lineal (LDA) y Bayes.  <br/>
<br/>
- **Ignorar la varianza baja:** Los conjuntos de datos a veces pueden contener características categóricas que tienen un número único o pequeño de valores en las muestras. Este tipo de características no solo no son informativas y no agregan valor, sino que a veces también son perjudiciales para algunos algoritmos. <br/>
<br/>
- **Multi-colinealidad:** Se da cuendo las variables tienen una correlacion muy alta. Algunos modelos se ven perjudicados por la multicolinealidad como los algoritmos lineales. La multicolinealidad puede reducir el coeficiente general del modelo y causar una variación impredecible. Esto conducirá a un sobreajuste donde el modelo puede funcionar muy bien en un conjunto de entrenamiento conocido, pero fallará con un conjunto de pruebas desconocido. <br/>
<br/>
- **Funciones de grupo:** A veces, los conjuntos de datos pueden contener funciones relacionadas a nivel de muestra. Por ejemplo, en el conjunto de datos `credit` hay características llamadas` BILL_AMT1 .. BILL_AMT6` que están relacionadas de tal manera que `BILL_AMT1` es la cantidad de la factura hace 1 mes y `BILL_AMT6` es la cantidad de la factura hace 6 meses. Estas características se pueden utilizar para extraer características adicionales basadas en las propiedades estadísticas de la distribución, como media, mediana, varianza, desviación estándar, etc. <br/>
<br/>
- **Discretización de Variables numéricas:** Binning o discretización es el proceso de transformar variables numéricas en características categóricas. Un ejemplo sería la variable Edad, que es una distribución continua de valores numéricos que se pueden discretizar en intervalos (10-20 años, 21-30, etc.). El agrupamiento puede mejorar la precisión de un modelo predictivo al reducir el ruido o la no linealidad en los datos. PyCaret determina automáticamente la cantidad y el tamaño de los contenedores usando la regla Sturges. <br/>
<br/>
- **Ensamblaje y apilamiento de modelos:** El modelado de conjuntos es un proceso en el que se crean varios modelos diversos para predecir un resultado. Esto se logra usando muchos algoritmos de modelado diferentes o usando diferentes muestras de conjuntos de datos de entrenamiento. Luego, el modelo de conjunto agrega las predicciones de cada modelo base, lo que da como resultado una predicción final para los datos. La motivación para usar modelos de conjuntos es reducir el error de generalización de la predicción. Siempre que los modelos base sean diversos e independientes, el error de predicción del modelo disminuye cuando se utiliza el enfoque de conjunto. Los dos métodos más comunes en el aprendizaje por conjuntos son "Bagging" y "Boosting". El apilamiento (Stacking) es también un tipo de aprendizaje conjunto en el que las predicciones de varios modelos se utilizan como características de entrada para un metamodelo que predice el resultado final.


## Importacion de los datos

You can download the data from the original source __[found here](https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients)__ and load it using the pandas read_csv function or you can use PyCaret's data respository to load the data using the get_data function (This will require an internet connection).

In [ ]:
from pycaret.datasets import get_data
dataset = get_data('credit')

In [ ]:
#!pip install pandas-profiling
from pandas_profiling import ProfileReport
import pandas as pd
data = pd.read_excel('data\default of credit card clients.xls', skiprows=1)
data.head()

data.profile_report()


- **Valores perdidos:** No hay valores perdidos en los datos. Sin embargo, todavía necesitamos imputadores en nuestro pipeline en caso de que los nuevos datos no vistos tengan valores faltantes (no aplicable en este caso). Cuando ejecuta la función `setup ()`, los imputadores se crean y almacenan en la tubería automáticamente. De forma predeterminada, utiliza un imputador medio para valores numéricos y un imputador constante para categóricos. Esto se puede cambiar usando los parámetros `numeric_imputation` e` categorical_imputation` en `setup ()`. <br/>
<br/>
- **Multicolinealidad:** Existen altas correlaciones entre `BILL_AMT1 ... BIL_AMT6` que genera multicolinealidad en los datos. Eliminaremos la multicolinealidad usando los parámetros `remove_multicollinearity` y` multicollinearity_threshold` en la configuración. <br/>
<br/>
- **Escala / rango de datos:** La escala de características numéricas son diferentes. Por ejemplo, la función `AGE` varía entre 21 y 79 y` BILL_AMT1` varía entre -165,580 y 964,511. Esto puede causar problemas para los algoritmos que asumen que todas las características tienen variaciones dentro del mismo orden. En este caso, el orden de magnitud de "BILL_AMT1" es muy diferente al de "AGE". Lo solucionaremos utilizando el parámetro `normalize` en la configuración. <br/>
<br/>
- **Distribución del espacio de funciones:** Las funciones numéricas no se distribuyen de forma normal, como: `LIMIT_BAL`,` BILL_AMT1` y `PAY_AMT1 ... PAY_AMT6`. Algunas características también están muy sesgadas, como `PAY_AMT1`. Esto puede causar problemas para los algoritmos que asumen distribuciones normales o aproximadas a la normal. Se soluciona utilizando el parámetro de transformación en la configuración. <br/>
<br/>
- **Funciones de grupo:** De la descripción de los datos sabemos que ciertas funciones están relacionadas entre sí, como `BILL_AMT1 ... BILL_AMT6` y` PAY_AMT1 ... PAY_AMT6`. Usaremos el parámetro `group_features` en la configuración para extraer información estadística de estas características. <br/>
<br/>
- **Funciones numéricas de Bin:** Al observar las correlaciones entre las características numéricas y la variable objetivo, vemos que "AGE" y "LIMIT_BAL" son débiles. Usaremos el parámetro `bin_numeric_features` para eliminar el ruido de estas variables que pueden ayudar a los algoritmos lineales. <br/>

In [ ]:
#check the shape of data
dataset.shape

Para demostrar la función `predict_model ()` en datos no vistos, se ha retenido una muestra de 1200 filas del conjunto de datos original para su uso en predicciones. Esto no debe confundirse con una división de train-test, ya que esta división en particular se realiza para simular un escenario de la vida real. 

In [ ]:
data = dataset.sample(frac=0.95, random_state=786)
data_unseen = dataset.drop(data.index)

data.reset_index(inplace=True, drop=True)
data_unseen.reset_index(inplace=True, drop=True)

print('Data for Modeling: ' + str(data.shape))
print('Unseen Data For Predictions ' + str(data_unseen.shape))

## Inicializar el entorno

In [ ]:
from pycaret.classification import *

In [ ]:
exp_clf102 = setup(data = data, target = 'default', session_id=123,
                  normalize = True, 
                  transformation = True, 
                  ignore_low_variance = True,
                  remove_multicollinearity = True, multicollinearity_threshold = 0.95,
                  bin_numeric_features = ['LIMIT_BAL', 'AGE'],
                  group_features = [['BILL_AMT1', 'BILL_AMT2','BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'],
                                   ['PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']],
                  log_experiment = True, experiment_name = 'credit1')

##  Comparacion de modelos

Utilizamos `compare_models()`para entrenar multiples modelos o identificar el mejor de ellos

In [ ]:
top3 = compare_models(n_select = 3)

Hemos utilizado el parámetro `n_select` dentro de` compare_models`.`compare_models` de forma predeterminada devuelve el modelo con mejor rendimiento (modelo único basado en el orden de clasificación predeterminado). Sin embargo, puedes usar el parámetro `n_select` para devolver los N mejores modelos. En este ejemplo, compare_models ha devuelto los 3 modelos principales.

In [ ]:
type(top3)

In [ ]:
print(top3)

Para fines de comparación, usaremos la puntuación `AUC`. Observa cuán drásticamente han mejorado algunos de los algoritmos después de que realizamos el preprocesamiento en `setup ()`.

- El AUC de regresión logística mejoró de `0.6514` a `0.7804`
- El AUC de Naives Bayes mejoró de `0.6458` a `0.7533`
- K AUC de vecinos más cercanos mejoró de `0.6097` a` 0.7004`



# 7.0 Create a Model

 En esta sección, crearemos todos los modelos usando 5 veces la validación cruzada. Observe cómo el parámetro `fold` se pasa dentro de` create_model () `para lograr esto.

### Modelos con 5 fold en la validacion cruzada

In [ ]:
dt = create_model('dt', fold = 5)

### Modelos redondeando las metricas a 2 decimales

In [ ]:
rf = create_model('rf', round = 2)

## Ajuste de hiperparametros

Para ajustar los hiperparametros utilizaremos`tune_model()`.En pycaret.classification, todo el ajuste de hiperparámetros está configurado para optimizar la precisión por defecto, lo que se puede cambiar utilizando el parámetro `optimize`

In [ ]:
tuned_rf = tune_model(rf, optimize = 'AUC')

In [ ]:
tuned_rf2 = tune_model(rf, optimize = 'Recall')

Los resultados de ajustar los clasificadores de random forest difieren con el parámetro `optimize`. En `tuned_rf` optimizamos` AUC` dando como resultado `0.7825` en` AUC` y `0.3445` en` Recall`. Sin embargo, en `tuned_rf2` cuando configuramos el parámetro `optimize` en `Recall`, resultó en un mejor modelo en términos de` Recall`. Sin embargo, "AUC" se vio comprometido. 

In [ ]:
plot_model(tuned_rf, plot = 'parameter')

In [ ]:
plot_model(tuned_rf2, plot = 'parameter')

## Ensamblado de modelos

El ensamble es una técnica común de aprendizaje automático que se utiliza para mejorar el rendimiento de los modelos (principalmente basados en árboles). Hay varias técnicas para ensamblar como Bagging y Boosting. Usaremos la función `ensemble_model ()` en PyCaret que ensambla los estimadores usando el método definido en el parámetro `method`.

In [ ]:
# lets create a simple decision tree model that we will use for ensembling 
dt = create_model('dt')

###  Bagging

In [ ]:
bagged_dt = ensemble_model(dt)

In [ ]:
# check the parameters of bagged_dt
print(bagged_dt)

El conjunto ha mejorado el "AUC" del clasificador de árbol de decisión. En el ejemplo anterior hemos usado los parámetros predeterminados de `ensemble_model ()` que usa el método `Bagging`. Probemos con `Boosting` cambiando el parámetro` method` en `ensemble_model ()`
 

###  Boosting

In [ ]:
boosted_dt = ensemble_model(dt, method = 'Boosting')

Observe lo fácil que es ensamblar modelos en PyCaret. Simplemente cambiando el parámetro "método", puede hacer un bagging o un boosting. Tenga en cuenta que `ensemble_model ()` construirá de forma predeterminada estimadores `10`. Esto se puede cambiar usando el parámetro `n_estimators`. Aumentar el número de estimadores a veces puede mejorar los resultados.

In [ ]:
bagged_dt2 = ensemble_model(dt, n_estimators=50)

Observe cómo el aumento del parámetro n_estimators ha mejorado el rendimiento del modelo ensamblado. El modelo bagged_dt con los estimadores predeterminados `10` resultó en un AUC de` 0.7345` mientras que en bagged_dt2 donde `n_estimators = 50` el AUC mejoró a` 0.7591`.

###  Blending

Blending o combinación es otra técnica común para ensamblar que se puede usar en PyCaret. Utiliza predicciones de múltiples modelos para generar un conjunto final de predicciones usando consenso de votación / mayoría de todos los modelos aprobados en el parámetro `estimator_list`. Si no se pasa ninguna lista, PyCaret usa todos los modelos disponibles en la biblioteca de modelos de forma predeterminada. El parámetro `método` se puede utilizar para definir el tipo de votación. Cuando se establece en "difícil", utiliza etiquetas para la votación de la regla de la mayoría. Cuando se establece en "suave", usa la suma de probabilidades predichas en lugar de la etiqueta

In [ ]:
blend_hard = blend_models()

Hemos creado un clasificador de votaciones usando la función `blend_models ()`. El modelo almacenado en la variable `blend_hard` es como cualquier otro modelo que crearía usando` create_model () `o` tune_model () `. Puede usar este modelo para predicciones sobre datos no vistos usando `predict_model ()` de la misma manera que lo haría con cualquier otro modelo. Tenga en cuenta que, dado que no pasamos la lista de modelos específicos para votar, utiliza todos los modelos de la biblioteca de modelos de forma predeterminada.

Es posible que haya notado que el `AUC` es cero para todos los pliegues. Esto se debe a que el parámetro "método" se establece en "difícil", que solo usa etiquetas (1 o 0) para las predicciones y, por lo tanto, no se calcula el AUC. Para cambiar esto, modificaría el parámetro `method` dentro de` blend_models () `

In [ ]:
blend_soft = blend_models(method = 'soft')

Los resultados no son muy diferentes. Los dos ejemplos anteriores utilizan todos los modelos de la biblioteca de modelos para crear un `VotingClassifier`. Normalmente, querrá combinar modelos específicos y no todos los modelos de la biblioteca. Vea el ejemplo a continuación donde combinamos los modelos `top3` obtenidos de` compare_models` como una lista. Simplemente pasaremos el parámetro `top3` dentro de` blend_models`.

In [ ]:
blend_top3 = blend_models(top3)

In [ ]:
print(blend_top3.estimators_)

### Stacking

El apilamiento o Stacking es otra técnica popular para ensamblar, pero se implementa con menos frecuencia debido a dificultades prácticas. El apilamiento es una técnica de aprendizaje por conjuntos que combina varios modelos a través de un metamodelo. Otra forma de pensar sobre el apilamiento es que se entrenan varios modelos para predecir el resultado y se crea un metamodelo que usa las predicciones de esos modelos como entrada junto con las características originales. 

In [ ]:
stack_soft = stack_models(top3)

In [ ]:
stack_hard = stack_models(top3, method='hard')

Similar a la combinación, `stack_models ()` también admite métodos blandos y duros que se pueden definir en el parámetro `method`. El método suave usa la suma de probabilidades predichas y el método difícil usa la etiqueta (1 o 0). En los dos ejemplos anteriores, el metamodelo (modelo final para generar predicciones) es Regresión logística (por defecto). El metamodelo se puede cambiar usando el parámetro `meta_model`

In [ ]:
xgboost = create_model('xgboost')
stack_soft2 = stack_models(top3, meta_model=xgboost)

La selección de qué "método" y "modelos" se utilizarán en el apilamiento depende de las propiedades estadísticas del conjunto de datos. Experimentar con diferentes modelos y métodos es la mejor manera de averiguar qué configuración funcionará mejor. Sin embargo, como regla general, los modelos con un rendimiento sólido pero diverso tienden a mejorar los resultados cuando se utilizan para apilar. </br>

Antes de terminar esta sección, hay otro parámetro en `stack_models ()` que aún no hemos visto llamado `restack`. Este parámetro controla la capacidad de exponer los datos sin procesar al metamodelo. Cuando se establece en "Verdadero", expone los datos sin procesar al metamodelo junto con todas las predicciones de los modelos de nivel base. De forma predeterminada, se establece en "Verdadero".

## Calibración de modelos

Al realizar la clasificación, a menudo no solo desea predecir la etiqueta de la clase (resultado como 0 o 1), sino también obtener la probabilidad del resultado respectivo que proporciona un nivel de confianza en la predicción. Algunos modelos pueden proporcionar estimaciones deficientes de las probabilidades de clase y algunos ni siquiera admiten la predicción de probabilidad. Los clasificadores bien calibrados son probabilísticos y proporcionan resultados en forma de probabilidades que pueden interpretarse directamente como un nivel de confianza. PyCaret le permite calibrar las probabilidades de un modelo dado a través de la función `calibrate_model ()`. 

In [ ]:
rf = create_model('rf')

In [ ]:
plot_model(rf, plot='calibration')

In [ ]:
calibrated_rf = calibrate_model(rf)

In [ ]:
plot_model(calibrated_rf, plot='calibration')

Observe cuán diferentes se ven los 2 gráficos anteriores. Uno es antes de la calibración y otro después. Un clasificador perfectamente calibrado seguirá la línea punteada negra en los gráficos anteriores. No solo `calibrated_rf` está mejor calibrado, sino que el` AUC` también ha mejorado de `0,7344` a` 0,7702`. Por defecto, `calibrate_model ()` usa el método `sigmoid` que corresponde al enfoque de Platt. El otro método disponible es "isotónico", que es un enfoque no paramétrico. Vea un ejemplo de calibración usando el método "isotónico" a continuación:

In [ ]:
calibrated_rf_isotonic = calibrate_model(rf, method = 'isotonic')

In [ ]:
plot_model(calibrated_rf_isotonic, plot='calibration')

La precisión de la muestra reservada es ** `0,8142` ** en comparación con los resultados del CV de ` 0,8204` en la sección 9.4 anterior. Tenga en cuenta que hay una disminución significativa en el "AUC" en el conjunto de reserva del CV

# Logging y MlFlow

PyCaret 2.0 incorpora el componente de seguimiento de MLflow como una API y una interfaz de usuario de backend para registrar parámetros, versiones de código, métricas y archivos de salida al ejecutar su código de aprendizaje automático y para visualizar posteriormente los resultados. Para registrar sus experimentos en pycaret, simplemente use los parámetros `log_experiment` y` experiment_name` en la función `setup`, como hicimos en este ejemplo.

Puede iniciar la interfaz de usuario en `localhost: 5000`. Simplemente inicie el servidor MLFlow desde la línea de comandos o desde la computadora portátil. Vea el ejemplo a continuación:

In [ ]:
# to start the MLFlow server from notebook:
!mlflow ui 

### ### Abra localhost: 5000 en su navegador (a continuación se muestra un ejemplo de cómo se ve la interfaz de usuario)
![title](https://i2.wp.com/pycaret.org/wp-content/uploads/2020/07/classification_mlflow_ui.png?resize=1080%2C508&ssl=1)